<p align="center">
  <img src="https://img.shields.io/badge/Axe%20AI%20Lab-TTS%20Studio-black?style=for-the-badge" />
  <img src="https://img.shields.io/badge/GPU-T4-green?style=for-the-badge&logo=nvidia" />
  <img src="https://img.shields.io/badge/Model-XTTS--v2-blue?style=for-the-badge" />
  <img src="https://img.shields.io/badge/Tunnel-Cloudflare-orange?style=for-the-badge&logo=cloudflare" />
</p>

---

# 🪓 Axe AI Lab — TTS Studio
### Powered by XTTS-v2 | Voice Cloning | 17 Languages | GPU Accelerated

> **Before running:** Go to `Runtime → Change runtime type → T4 GPU` and save.

| Step | Cell | Description | Time |
|------|------|-------------|------|
| 1 | GPU Check | Verify T4 is active | ~5s |
| 2 | Install | All packages + cloudflared | ~2 min |
| 3 | Load Model | Download & init XTTS-v2 | ~3-5 min (first run) |
| 4 | Launch UI | Start Gradio app | ~10s |
| 5 | Tunnel | Get public Cloudflare URL | ~10s |


In [ ]:
# ============================================================
#  CELL 1 — GPU CHECK
#  Make sure you selected T4 GPU in Runtime settings!
# ============================================================

import subprocess, torch

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
    print(f'PyTorch CUDA available : {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        print(f'GPU Name               : {torch.cuda.get_device_name(0)}')
        print(f'VRAM                   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU detected!')
    print('Go to: Runtime > Change runtime type > T4 GPU > Save')


In [ ]:
# ============================================================
#  CELL 2 — INSTALL DEPENDENCIES
#  Uses Kokoro-82M: works on Python 3.12, fast on T4 GPU
# ============================================================

import sys
print(f'Python : {sys.version.split()[0]}')

print('[1/5] Installing Kokoro TTS...')
!pip install -q kokoro>=0.9.4 soundfile

print('[2/5] Installing espeak-ng (phonemizer backend)...')
!apt-get install -qq espeak-ng

print('[3/5] Installing Gradio...')
!pip install -q "gradio>=4.44.0"

print('[4/5] Installing audio utilities...')
!pip install -q pydub scipy numpy
!apt-get install -qq ffmpeg

print('[5/5] Installing Cloudflare tunnel...')
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# Sanity check
import importlib
print()
for pkg, mod in [('kokoro','kokoro'), ('gradio','gradio'), ('torch','torch'), ('soundfile','soundfile')]:
    try:
        m = importlib.import_module(mod)
        ver = getattr(m, '__version__', 'ok')
        print(f'  ✅ {pkg}: {ver}')
    except Exception as e:
        print(f'  ❌ {pkg}: FAILED — {e}')


In [ ]:
# ============================================================
#  CELL 3 — LOAD KOKORO-82M MODEL
#  Downloads ~330MB on first run, cached after.
#  Supports: en-us, en-gb, ja, zh, es, fr, hi, it, pt, ko
# ============================================================

import torch, time
from kokoro import KPipeline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device.upper()}')
if device == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

# Kokoro voice packs — lang_code maps to pipeline
# One pipeline per language family; we'll lazy-load others on demand
print('\nLoading Kokoro en-us pipeline (downloads model ~330MB first run)...')
t0 = time.time()

pipelines = {}   # cache: lang_code -> KPipeline

def get_pipeline(lang_code='a'):  # 'a'=en-us, 'b'=en-gb
    if lang_code not in pipelines:
        pipelines[lang_code] = KPipeline(lang_code=lang_code)
    return pipelines[lang_code]

# Pre-load English
get_pipeline('a')
print(f'✅ Kokoro loaded in {time.time()-t0:.1f}s')

# Available voices per language
VOICE_MAP = {
    'English (US)'    : ('a', ['af_heart', 'af_bella', 'af_sarah', 'am_adam', 'am_michael']),
    'English (UK)'    : ('b', ['bf_emma', 'bf_isabella', 'bm_george', 'bm_lewis']),
    'Japanese'        : ('j', ['jf_alpha', 'jf_gongitsune', 'jm_kumo']),
    'Chinese'         : ('z', ['zf_xiaobei', 'zf_xiaoni', 'zm_yunjian']),
    'Spanish'         : ('e', ['ef_dora', 'em_alex', 'em_santa']),
    'French'          : ('f', ['ff_siwis']),
    'Hindi'           : ('h', ['hf_alpha', 'hf_beta', 'hm_omega']),
    'Italian'         : ('i', ['if_sara', 'im_nicola']),
    'Brazilian PT'    : ('p', ['pf_dora', 'pm_alex', 'pm_santa']),
    'Korean'          : ('k', ['kf_alpha', 'km_omega']),
}

print(f'\nLanguages ready  : {list(VOICE_MAP.keys())}')
print(f'Default voices   : {VOICE_MAP["English (US)"][1]}')


In [ ]:
# ============================================================
#  CELL 4 — AXE AI LAB TTS STUDIO (GRADIO UI)
#  Kokoro-82M | 10 languages | multiple voices | T4 GPU
# ============================================================

import gradio as gr
import numpy as np
import soundfile as sf
import tempfile, os, time
from datetime import datetime
import torch

OUTPUT_DIR = '/tmp/axe_tts_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

history_log = []

def get_voices_for_lang(language):
    if language not in VOICE_MAP:
        return [], None
    _, voices = VOICE_MAP[language]
    return gr.update(choices=voices, value=voices[0])

def generate_speech(text, language, voice, speed):
    if not text or not text.strip():
        return None, '⚠️ Please enter some text.', format_history()

    lang_code, _ = VOICE_MAP.get(language, ('a', []))
    timestamp = datetime.now().strftime('%H%M%S_%f')[:10]
    out_path = os.path.join(OUTPUT_DIR, f'axe_{timestamp}.wav')

    try:
        t0 = time.time()
        pipe = get_pipeline(lang_code)

        # Kokoro returns generator of (graphemes, phonemes, audio_tensor)
        audio_chunks = []
        for _, _, audio in pipe(text, voice=voice, speed=speed):
            if audio is not None:
                audio_chunks.append(audio.numpy() if hasattr(audio, 'numpy') else np.array(audio))

        if not audio_chunks:
            return None, '❌ No audio generated. Try different text or voice.', format_history()

        full_audio = np.concatenate(audio_chunks)
        sf.write(out_path, full_audio, 24000)

        elapsed = time.time() - t0
        entry = {
            'time': timestamp[:6], 'lang': language, 'voice': voice,
            'chars': len(text), 'duration': f'{elapsed:.1f}s',
            'preview': text[:55] + ('...' if len(text) > 55 else ''),
        }
        history_log.insert(0, entry)
        if len(history_log) > 5:
            history_log.pop()

        vram = ''
        if torch.cuda.is_available():
            used = torch.cuda.memory_allocated() / 1e9
            total = torch.cuda.get_device_properties(0).total_memory / 1e9
            vram = f'  |  VRAM {used:.1f}/{total:.1f}GB'

        status = f'✅ Done in {elapsed:.1f}s  |  {language}  |  {voice}  |  {len(text)} chars  |  {speed}x speed{vram}'
        return out_path, status, format_history()

    except Exception as e:
        import traceback
        return None, f'❌ Error: {str(e)}\n{traceback.format_exc()[-300:]}', format_history()


def format_history():
    if not history_log:
        return 'No generations yet.'
    return '\n'.join([
        f"[{h['time']}] {h['lang']} | {h['voice']} | {h['chars']} chars | {h['duration']} | \'{h['preview']}\'"
        for h in history_log
    ])


CSS = """
@import url('https://fonts.googleapis.com/css2?family=Syne:wght@400;700;800&family=DM+Mono:wght@400;500&display=swap');
body, .gradio-container { background: #0a0a0a !important; font-family: 'DM Mono', monospace !important; }
#axe-header { background: #0a0a0a; border-bottom: 1px solid #1e1e1e; padding: 28px 0 20px; text-align: center; margin-bottom: 8px; }
#axe-header h1 { font-family: 'Syne', sans-serif !important; font-size: 2.6rem; font-weight: 800; letter-spacing: -1px; color: #fff; margin: 0; }
#axe-header h1 span { color: #ff3c00; }
#axe-header p { font-size: 0.72rem; color: #555; letter-spacing: 3px; text-transform: uppercase; margin: 6px 0 0; }
.axe-badge { display: inline-block; background: #111; border: 1px solid #222; color: #666; font-size: 0.62rem; letter-spacing: 2px; padding: 3px 10px; border-radius: 2px; margin: 0 3px; text-transform: uppercase; }
label, .block > label > span { color: #777 !important; font-size: 0.68rem !important; letter-spacing: 2px !important; text-transform: uppercase !important; font-family: 'DM Mono', monospace !important; }
textarea, .dropdown { background: #111 !important; border: 1px solid #1e1e1e !important; color: #ddd !important; border-radius: 4px !important; font-family: 'DM Mono', monospace !important; }
textarea:focus { border-color: #ff3c00 !important; box-shadow: 0 0 0 2px rgba(255,60,0,0.12) !important; }
button.primary { background: #ff3c00 !important; border: none !important; color: white !important; font-family: 'Syne', sans-serif !important; font-weight: 700 !important; letter-spacing: 2px !important; text-transform: uppercase !important; border-radius: 3px !important; }
button.primary:hover { background: #cc3000 !important; }
.block { background: #0e0e0e !important; border: 1px solid #181818 !important; border-radius: 6px !important; }
.tab-nav button { font-family: 'DM Mono', monospace !important; font-size: 0.68rem !important; letter-spacing: 2px !important; text-transform: uppercase !important; color: #555 !important; }
.tab-nav button.selected { color: #ff3c00 !important; border-bottom: 2px solid #ff3c00 !important; }
footer { display: none !important; }
"""

with gr.Blocks(title='Axe AI Lab — TTS Studio', css=CSS,
    theme=gr.themes.Base(primary_hue='orange', neutral_hue='gray',
    font=gr.themes.GoogleFont('DM Mono'))) as demo:

    gr.HTML("""
    <div id="axe-header">
        <h1>🪓 AXE <span>AI</span> LAB</h1>
        <p>TTS Studio &nbsp;·&nbsp; Kokoro-82M &nbsp;·&nbsp; GPU Accelerated</p>
        <div style="margin-top:14px">
            <span class="axe-badge">Kokoro-82M</span>
            <span class="axe-badge">10 Languages</span>
            <span class="axe-badge">T4 GPU</span>
            <span class="axe-badge">24kHz Output</span>
        </div>
    </div>
    """)

    with gr.Tabs():

        # ── Generate tab ─────────────────────────────────────
        with gr.Tab("Generate"):
            with gr.Row():
                with gr.Column(scale=3):
                    text_input = gr.Textbox(
                        label="Text to Synthesize",
                        lines=7, max_lines=20,
                        placeholder="Type or paste your text here...\n\nKokoro handles long-form text naturally with correct prosody and rhythm."
                    )
                    with gr.Row():
                        language = gr.Dropdown(
                            choices=list(VOICE_MAP.keys()),
                            value='English (US)',
                            label='Language',
                            scale=2,
                        )
                        voice = gr.Dropdown(
                            choices=VOICE_MAP['English (US)'][1],
                            value=VOICE_MAP['English (US)'][1][0],
                            label='Voice',
                            scale=2,
                        )
                    speed = gr.Slider(minimum=0.5, maximum=2.0, value=1.0, step=0.05, label='Speed')

                    language.change(get_voices_for_lang, inputs=[language], outputs=[voice])

                    generate_btn = gr.Button("Generate Speech", variant="primary", size="lg")

                with gr.Column(scale=2):
                    audio_output = gr.Audio(label='Output Audio', type='filepath', interactive=False)
                    status_box = gr.Textbox(label='Status', interactive=False, lines=3)
                    gr.HTML("""
                    <div style="margin-top:16px; padding:16px; background:#111; border:1px solid #1e1e1e; border-radius:4px;">
                        <p style="color:#444; font-size:0.62rem; letter-spacing:2px; text-transform:uppercase; margin:0 0 10px;">Kokoro voices guide</p>
                        <ul style="color:#555; font-size:0.72rem; line-height:1.9; margin:0; padding-left:16px;">
                            <li><b style="color:#666">af_heart / af_bella</b> — warm female US</li>
                            <li><b style="color:#666">am_adam / am_michael</b> — male US</li>
                            <li><b style="color:#666">bf_emma / bm_george</b> — British accents</li>
                            <li><b style="color:#666">hf_alpha / hm_omega</b> — Hindi voices</li>
                            <li>Speed 0.9 often sounds most natural</li>
                        </ul>
                    </div>
                    """)

            _h = gr.Textbox(visible=False)
            generate_btn.click(
                generate_speech,
                inputs=[text_input, language, voice, speed],
                outputs=[audio_output, status_box, _h]
            )

        # ── History tab ──────────────────────────────────────
        with gr.Tab("History"):
            gr.Markdown("### Last 5 Generations")
            history_display = gr.Textbox(value='No generations yet.', label='Log', lines=10, interactive=False)
            gr.Button("Refresh", variant="secondary").click(lambda: format_history(), outputs=[history_display])

        # ── Examples tab ─────────────────────────────────────
        with gr.Tab("Examples"):
            gr.Markdown("### Quick-start examples")
            gr.Examples(
                examples=[
                    ["Welcome to Axe AI Lab. We build fast, powerful, and open AI tools for everyone.", "English (US)", "af_heart", 1.0],
                    ["नमस्ते! Axe AI Lab में आपका स्वागत है। हम सबके लिए बेहतरीन AI टूल्स बनाते हैं।", "Hindi", "hf_alpha", 1.0],
                    ["Bonjour! Bienvenue sur Axe AI Lab. Nous construisons des outils d\'IA pour tous.", "French", "ff_siwis", 0.95],
                    ["人工知能は私たちの生活を大きく変えています。Axe AI Labへようこそ。", "Japanese", "jf_alpha", 1.0],
                    ["The quick brown fox jumps over the lazy dog. She sells seashells by the seashore.", "English (UK)", "bm_george", 0.9],
                ],
                inputs=[text_input, language, voice, speed],
            )

        # ── About tab ────────────────────────────────────────
        with gr.Tab("About"):
            gr.Markdown("""
## 🪓 Axe AI Lab — TTS Studio
Built on **Kokoro-82M** — a lightweight but high-quality open TTS model.

### Model Specs
| Property | Value |
|----------|-------|
| Model | Kokoro-82M |
| Parameters | 82 Million |
| Sample Rate | 24 kHz |
| Languages | 10 |
| Python Support | 3.9 – 3.12 |
| License | Apache 2.0 |

### Languages & Voices
| Language | Code | Voices |
|----------|------|--------|
| English US | a | af_heart, af_bella, af_sarah, am_adam, am_michael |
| English UK | b | bf_emma, bf_isabella, bm_george, bm_lewis |
| Japanese | j | jf_alpha, jf_gongitsune, jm_kumo |
| Chinese | z | zf_xiaobei, zf_xiaoni, zm_yunjian |
| Spanish | e | ef_dora, em_alex, em_santa |
| French | f | ff_siwis |
| Hindi | h | hf_alpha, hf_beta, hm_omega |
| Italian | i | if_sara, im_nicola |
| Portuguese BR | p | pf_dora, pm_alex |
| Korean | k | kf_alpha, km_omega |

---
*Axe AI Lab · Kokoro-82M · T4 GPU · Cloudflare Tunnel*
            """)

    gr.HTML("""
    <div style="text-align:center; padding:20px 0 8px; border-top:1px solid #181818; margin-top:20px;">
        <span style="font-size:0.62rem; color:#2a2a2a; letter-spacing:3px; text-transform:uppercase;">
            Axe AI Lab &nbsp;·&nbsp; TTS Studio &nbsp;·&nbsp; Kokoro-82M &nbsp;·&nbsp; T4 GPU
        </span>
    </div>
    """)

print('🪓 Launching Axe AI Lab TTS Studio...')
demo.launch(server_port=7860, share=False, quiet=True, show_error=True)
print('✅ Gradio running on http://localhost:7860')


In [ ]:
# ============================================================
#  CELL 5 — CLOUDFLARE TUNNEL
#  Run AFTER Cell 4 is fully running.
#  Gives you a public URL valid for your entire Colab session.
# ============================================================

import subprocess, threading, time, re

PUBLIC_URL = None

def run_tunnel():
    global PUBLIC_URL
    proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://localhost:7860'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    for line in proc.stdout:
        decoded = line.decode('utf-8', errors='ignore')
        match = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', decoded)
        if match:
            PUBLIC_URL = match.group(0)
            break

threading.Thread(target=run_tunnel, daemon=True).start()

print('Starting Cloudflare tunnel...')
for i in range(25):
    time.sleep(1)
    if PUBLIC_URL:
        break
    print(f'  waiting... {i+1}s', end='\r')

if PUBLIC_URL:
    print('\n')
    print('=' * 58)
    print('  🪓  AXE AI LAB — TTS STUDIO')
    print('=' * 58)
    print(f'  PUBLIC URL : {PUBLIC_URL}')
    print('=' * 58)
    print('  Share this link — opens from any device!')
    print('  Active while this Colab session is running.')
    print('=' * 58)
else:
    print('\n⚠️  Could not get tunnel URL. Re-run this cell.')
    print('   Local access: http://localhost:7860')
